In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df=pd.read_json('../original_datasets/customer_reviews.jsonl', lines=True)
print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 12150 entries, 0 to 12149
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   review_id                  12150 non-null  str    
 1   order_id                   11903 non-null  str    
 2   product_id                 12150 non-null  str    
 3   customer_id                12150 non-null  str    
 4   rating                     12150 non-null  object 
 5   review_title               9716 non-null   str    
 6   review_text                11678 non-null  str    
 7   review_date                12150 non-null  str    
 8   verified_purchase          12150 non-null  object 
 9   helpful_votes              8153 non-null   float64
 10  delivery_mode_experienced  10129 non-null  str    
 11  sentiment_label            7885 non-null   str    
dtypes: float64(1), object(2), str(9)
memory usage: 1.1+ MB
None
    review_id    order_id product_id customer_id rati

In [ ]:
# 1. Check for total duplicate rows across the entire dataset
total_duplicates = df.duplicated().sum()
print(f"Total fully duplicated rows: {total_duplicates}")

# 2. Check for duplicate review_IDs (since each review should have a unique identifier)
duplicate_reviews = df['review_id'].duplicated().sum()
print(f"Number of duplicate review_ids: {duplicate_reviews}")

# 3. View the duplicated review_ids if any exist
if duplicate_reviews > 0:
    print("\nSample of duplicate review_ids:")
    print(df[df['review_id'].duplicated(keep=False)][['review_id', 'product_id', 'customer_id']].head(10))
#Drop duplicates based strictly on 'review_id', keeping the first occurren
df = df.drop_duplicates(subset=['review_id'], keep='first').copy()

Total fully duplicated rows: 150
Number of duplicate review_ids: 150

Sample of duplicate review_ids:
      review_id product_id customer_id
31   REV-011020  PRD-00116  CUST-01619
35   REV-006039  PRD-00474  CUST-03188
132  REV-001380  PRD-00784  CUST-00621
170  REV-000448  PRD-00473  CUST-03074
220  REV-009619  PRD-00157  CUST-02090
255  REV-006692  PRD-00614  CUST-02628
332  REV-006397  PRD-00734  CUST-02985
425  REV-004324  PRD-00429  CUST-00364
453  REV-009351  PRD-00533  CUST-00291
597  REV-002600  PRD-00579  CUST-02566


In [6]:
df = df.drop_duplicates(subset=['review_id'], keep='first').copy()

In [8]:
print(df.isna().sum())

review_id                       0
order_id                      246
product_id                      0
customer_id                     0
rating                          0
review_title                 2403
review_text                   464
review_date                     0
verified_purchase               0
helpful_votes                3947
delivery_mode_experienced    2001
sentiment_label              4217
dtype: int64


In [12]:
#1. Make a copy to work on
df_reviews_clean = df.copy()

def clean_rating(val):
    if pd.isna(val):
        return None
    val_str = str(val).lower().strip()
    word_map = {
        'one': 1, '1': 1, '1/5': 1, '1 star': 1, '1 stars': 1,
        'two': 2, '2': 2, '2/5': 2, '2 stars': 2,
        'three': 3, '3': 3, '3/5': 3, '3 stars': 3,
        'four': 4, '4': 4, '4/5': 4, '4 stars': 4,
        'five': 5, '5': 5, '5/5': 5, '5 stars': 5
    }
    for key, num in word_map.items():
        if key in val_str:
            return num
    digits = re.findall(r'\d+', val_str)
    if digits:
        return int(digits[0])
    return None

df_reviews_clean['rating'] = df_reviews_clean['rating'].apply(clean_rating)

# 3. Standardize verified_purchase column to boolean
df_reviews_clean['verified_purchase'] = (
    df_reviews_clean['verified_purchase']
    .astype(str)
    .str.lower()
    .map({'true': True, 'false': False, '1': True, '0': False})
    .fillna(False)
)

# 4. Fill missing text/metadata fields with sensible defaults
df_reviews_clean['review_title'] = df_reviews_clean['review_title'].fillna('No Title')
df_reviews_clean['review_text'] = df_reviews_clean['review_text'].fillna('')
df_reviews_clean['delivery_mode_experienced'] = df_reviews_clean['delivery_mode_experienced'].fillna('Unknown')

# 5. Fill missing helpful votes with 0 (assuming NaN means zero recorded votes)
df_reviews_clean['helpful_votes'] = df_reviews_clean['helpful_votes'].fillna(0)

# 6. Drop rows where critical identifiers or text are completely missing
df_reviews_clean = df_reviews_clean.dropna(subset=['review_id', 'product_id', 'customer_id'])

print("Customer reviews successfully cleaned and filled!")
print(f"Total rows after cleaning: {len(df_reviews_clean)}")
print(df_reviews_clean[['review_id', 'rating', 'verified_purchase', 'helpful_votes', 'delivery_mode_experienced']].head())

# Optional: Save cleaned reviews
# df_reviews_clean.to_csv('cleaned_customer_reviews.csv', index=False)

Customer reviews successfully cleaned and filled!
Total rows after cleaning: 12000
    review_id  rating  verified_purchase  helpful_votes  \
0  REV-000178       5              False            0.0   
1  REV-001266       1              False            0.0   
2  REV-008142       3              False           63.0   
3  REV-008108       3               True           78.0   
4  REV-008239       5              False           17.0   

  delivery_mode_experienced  
0                       Van  
1                   Unknown  
2                     drone  
3                     Drone  
4                       Van  


In [17]:
print(df_reviews_clean.isna().sum())

review_id                       0
order_id                      246
product_id                      0
customer_id                     0
rating                          0
review_title                    0
review_text                     0
review_date                     0
verified_purchase               0
helpful_votes                   0
delivery_mode_experienced       0
sentiment_label              4217
dtype: int64


In [19]:
# Function to infer sentiment from the numeric rating if sentiment_label is missing
def impute_sentiment(row):
    if pd.notna(row['sentiment_label']):
        return row['sentiment_label']
    
    rating = row['rating']
    if rating >= 4:
        return 'pos'
    elif rating == 3:
        return 'neu'
    elif rating <= 2:
        return 'neg'
    return 'neu' # fallback

# Apply the imputation to fill missing sentiment labels
df_reviews_clean['sentiment_label'] = df_reviews_clean.apply(impute_sentiment, axis=1)

# Also check for any missing order_ids if needed
df_reviews_clean['order_id'] = df_reviews_clean['order_id'].fillna('ORD-UNKNOWN')

# Verify that all nulls are gone
print(df_reviews_clean.isnull().sum())

review_id                    0
order_id                     0
product_id                   0
customer_id                  0
rating                       0
review_title                 0
review_text                  0
review_date                  0
verified_purchase            0
helpful_votes                0
delivery_mode_experienced    0
sentiment_label              0
dtype: int64


In [18]:
# Standardize delivery mode casing (e.g., 'drone' and 'Drone' become 'Drone')
df_reviews_clean['delivery_mode_experienced'] = df_reviews_clean['delivery_mode_experienced'].str.title()

print(df_reviews_clean['delivery_mode_experienced'].value_counts())

delivery_mode_experienced
Drone      3992
Van        2058
Unknown    2001
Truck      1981
Bike       1968
Name: count, dtype: int64


In [20]:
# Check for outliers in helpful_votes
Q1_votes = df_reviews_clean['helpful_votes'].quantile(0.25)
Q3_votes = df_reviews_clean['helpful_votes'].quantile(0.75)
IQR_votes = Q3_votes - Q1_votes

vote_outliers = df_reviews_clean[
    (df_reviews_clean['helpful_votes'] < (Q1_votes - 1.5 * IQR_votes)) | 
    (df_reviews_clean['helpful_votes'] > (Q3_votes + 1.5 * IQR_votes))
]

print(f"Number of helpful_votes outliers found: {len(vote_outliers)}")
print(df_reviews_clean['helpful_votes'].describe())

Number of helpful_votes outliers found: 0
count    12000.000000
mean        30.199750
std         30.196142
min          0.000000
25%          0.000000
50%         23.000000
75%         57.000000
max         90.000000
Name: helpful_votes, dtype: float64


In [21]:
df_reviews_clean.to_csv('cleaned_customer_reviews.csv', index=False)